# Offline parsing with a small local LM — variant E

Builds a parse cache with a small local language model, so that **retraining never runs a
parser**. Training reads a precomputed `tokenized_parsed_result_{split}.json`, exactly as
it does for the GPT-4o-mini and LLaMA caches already in `data_parsing/`.

The point of variant E (Reviewer #3 comment 1): if a 0.5B model that runs on your own
machine lands close to GPT-4o-mini, the API leaves the method's critical path entirely,
which also answers the cost objection.

**One model per run, one folder per model.** `parse_with_smalllm.py` derives the output
folder from the model name, so parsing with `qwen2.5` and then `smollm2` produces two
independent caches instead of the second silently overwriting the first.

| Stage | Cell | Cost |
|---|---|---|
| Pick a model | 1 | free — downloads nothing |
| Smoke test (8 descriptions) | 2 | ~1 min, downloads the model |
| Full parse (train + val) | 3 | 20–60 min on GPU, hours on CPU |
| Inspect the output | 4–6 | seconds |
| Score it against the others | 7 | seconds |

> **GPU strongly recommended.** On CPU a 0.5B model produces roughly 1–2 descriptions/s,
> and the two splits together are 46,173 descriptions. The run checkpoints every 500
> records and resumes automatically, so interrupting it is safe.

---
## 0 · Setup

In [ ]:
import os

# EDIT THIS if the notebook is not already sitting in the repository root.
REPO = "/home/ario/PY/3D-VG"

os.chdir(REPO)
print("cwd:", os.getcwd())

In [ ]:
# COLAB ONLY -- skip when running locally.
RUN_COLAB_SETUP = False

if RUN_COLAB_SETUP:
    from google.colab import drive
    drive.mount('/content/drive')

    %cd /content/drive/MyDrive/3D-VG

    !pip install -q 'transformers>=4.40' accelerate sentencepiece

    import os
    print("cwd:", os.getcwd())

### 0b · Preflight

Checks the inputs and reports whether a GPU is genuinely usable. `torch.cuda.is_available()`
is not sufficient on its own — a card older than the kernels the installed torch was built
for reports `True` and then fails on the first real operation, so this runs an actual
matmul.

In [ ]:
import json, os, sys

print(f"python {sys.version.split()[0]}   cwd {os.getcwd()}\n")

for split in ("train", "val"):
    path = f"data/ScanRefer_filtered_{split}.json"
    if os.path.isfile(path):
        print(f"  OK       {path}  ({len(json.load(open(path)))} descriptions)")
    else:
        print(f"  MISSING  {path}")

try:
    import transformers
    print(f"  OK       transformers {transformers.__version__}")
except ImportError:
    print("  MISSING  transformers   -> pip install 'transformers>=4.40' accelerate")

sys.path.insert(0, os.getcwd())
from experiments.ablation.parsers.run_smalllm_parser import cuda_is_usable

usable, detail = cuda_is_usable()
DEVICE = "cuda" if usable else "cpu"
DTYPE = "float16" if usable else "float32"
print(f"  {'OK      ' if usable else 'NO GPU  '} {detail}")
print(f"\n-> will run on {DEVICE} ({DTYPE})")
if not usable:
    print("   CPU works but is slow. Use --limit for a smoke test, or attach a GPU.")

---
## 1 · Which model?

Four small models are registered. This prints the table and **downloads nothing** — the
`needs` column is checked against what is actually installed, so a missing dependency
surfaces here rather than after a 1 GB download.

In [ ]:
!python experiments/ablation/parsers/parse_with_smalllm.py --list-models

Pick one below. `qwen2.5` is the default and has the best JSON adherence of the four,
which matters because the malformed-output rate is a number the manuscript has to report.

In [ ]:
MODEL = "qwen2.5"          # qwen2.5 | qwen2 | flan-t5 | smollm2, or any HuggingFace id

from experiments.ablation.parsers.parse_with_smalllm import folders_for

RAW_DIR, TOK_DIR = folders_for(MODEL)
PARSING_FOLDER = os.path.basename(TOK_DIR)

print(f"model          : {MODEL}")
print(f"raw phrases    : {RAW_DIR}/")
print(f"training cache : {TOK_DIR}/")
print(f"\n--parsing_folder {PARSING_FOLDER}")

---
## 2 · Smoke test — 8 descriptions

Run this **before** the full parse. It downloads the model (~1 GB on a cold cache) and
proves the whole path works end to end: prompt, generation, JSON extraction, tokenisation,
the 7/17/75 caps, and the schema check.

It writes into the same folders as the real run, and the full run in section 3 simply
overwrites them.

In [ ]:
!python experiments/ablation/parsers/parse_with_smalllm.py \
    --model {MODEL} --splits val --limit 8 \
    --device {DEVICE} --dtype {DTYPE} --batch-size 4

### 2b · What the smoke test produced

`ok` / `repaired` / `malformed` is the output-quality signal. `repaired` means the model
emitted almost-JSON (single quotes, a trailing comma) and it was fixed; `malformed` means
nothing usable came back and all three fields fell back to `"not mentioned"`.

In [ ]:
from IPython.display import HTML, Markdown, display

summary_path = os.path.join(TOK_DIR, "parse_run_summary.json")
if os.path.isfile(summary_path):
    S = json.load(open(summary_path))
    display(Markdown(f"**model** `{S['model']}` · **{S['kind']}** · device `{S['device']}` "
                     f"· dtype `{S['dtype']}` · greedy `{S['greedy']}`"))
    rows = []
    for split, st in S["splits"].items():
        generated = max(st["generated"], 1)
        rows.append([split, st["records"], st["ok"], st["repaired"], st["malformed"],
                     f"{100 * st['malformed'] / generated:.1f}%",
                     f"{st['desc_per_sec']:.2f}/s"])
    html = ["<table style='border-collapse:collapse;font-size:13px'>",
            "<tr>" + "".join(f"<th style='border-bottom:2px solid #444;padding:4px 10px'>{h}</th>"
                             for h in ("split", "records", "ok", "repaired", "malformed",
                                       "malformed %", "speed")) + "</tr>"]
    for r in rows:
        html.append("<tr>" + "".join(
            f"<td style='border-bottom:1px solid #ddd;padding:4px 10px'>{c}</td>"
            for c in r) + "</tr>")
    html.append("</table>")
    display(HTML("".join(html)))
else:
    print(f"{summary_path} not found -- did the smoke test run?")

### 2c · The actual parses, so you can see what the model returns

In [ ]:
tok_path = os.path.join(TOK_DIR, "tokenized_parsed_result_val.json")
raw_path = os.path.join(RAW_DIR, "parsed_result_val.json")

if os.path.isfile(tok_path):
    tok = json.load(open(tok_path))
    raw = json.load(open(raw_path)) if os.path.isfile(raw_path) else {}
    descriptions = {f"{r['scene_id']}|{r['object_id']}|{r['ann_id']}": r["description"]
                    for r in json.load(open("data/ScanRefer_filtered_val.json"))}

    shown = 0
    for scene, objs in tok.items():
        for obj, anns in objs.items():
            for ann, fields in anns.items():
                key = f"{scene}|{obj}|{ann}"
                display(Markdown(f"**{key}**"))
                display(HTML(f"<div style='padding:5px 9px;border-left:3px solid #888;"
                             f"background:#FAFAFA;font-style:italic;font-size:12.5px'>"
                             f"{descriptions.get(key, '')}</div>"))
                rows = []
                for field in ("target", "adjectives", "neighbors"):
                    tokens = fields[field]
                    empty = tokens == ["not", "mentioned"]
                    rows.append((field, " ".join(tokens), len(tokens), empty))
                html = ["<table style='border-collapse:collapse;font-size:12.5px;"
                        "margin-bottom:10px'>"]
                for field, text, n, empty in rows:
                    colour = "color:#B00020;background:#FFF4F4;" if empty else ""
                    html.append(
                        f"<tr><td style='padding:3px 9px;font-weight:bold'>{field}</td>"
                        f"<td style='padding:3px 9px;font-family:monospace;{colour}'>{text}</td>"
                        f"<td style='padding:3px 9px;color:#888'>{n} tok</td></tr>")
                html.append("</table>")
                display(HTML("".join(html)))
                shown += 1
                if shown >= 4:
                    break
            if shown >= 4:
                break
        if shown >= 4:
            break
else:
    print("no tokenized output yet -- run the smoke test above first")

---
## 3 · The full parse — train + val

**This is the long one.** 46,173 descriptions across both splits.

- on a T4 GPU: roughly 20–60 minutes depending on the model
- on CPU: hours

It checkpoints every 500 records into `.partial_{split}.json` and resumes automatically,
so a disconnected Colab session costs only the work since the last checkpoint. The
checkpoint records which model wrote it and **refuses to resume across a model change**,
so switching `MODEL` cannot silently blend two models' output into one cache.

Set `RUN_FULL_PARSE = True` to start. It is off by default so *Run All* cannot launch an
hours-long job by accident.

In [ ]:
RUN_FULL_PARSE = False

if RUN_FULL_PARSE:
    !python experiments/ablation/parsers/parse_with_smalllm.py \
        --model {MODEL} --splits train val \
        --device {DEVICE} --dtype {DTYPE} --batch-size 32
else:
    print("RUN_FULL_PARSE is False -- set it to True to start the real parse.")
    print(f"or run it in a terminal, which survives a browser disconnect:\n")
    print(f"    python experiments/ablation/parsers/parse_with_smalllm.py \\")
    print(f"        --model {MODEL} --splits train val --device {DEVICE}")

---
## 4 · Verify the cache before training on it

Training reads these files directly, and a violation raises **mid-epoch**, not at load —
`lib/dataset.py` allocates fixed `(7|17|75, 300)` arrays but passes the *unclipped* token
count to `pack_padded_sequence`. So check here, where it costs a second.

In [ ]:
from experiments.ablation.parsers.tokenize_parse import (
    FIELD_MAX_TOKENS, validate_tokenized)

for split in ("train", "val"):
    path = os.path.join(TOK_DIR, f"tokenized_parsed_result_{split}.json")
    if not os.path.isfile(path):
        print(f"  ABSENT   {path}")
        continue
    data = json.load(open(path))
    ok, problems, checked = validate_tokenized(data, label=f"{split}:")
    size = os.path.getsize(path) / (1024 ** 2)
    if ok:
        print(f"  OK       {split:5s} {checked:6d} entries, {size:.1f} MB "
              f"(keys, non-empty, within caps, all strings)")
    else:
        print(f"  FAILED   {split:5s} {len(problems)} problems:")
        for problem in problems[:10]:
            print(f"             {problem}")

print(f"\ncaps: " + "  ".join(f"{f}={n}" for f, n in FIELD_MAX_TOKENS.items()))

---
## 5 · Field statistics — what did the model actually produce?

Coverage per field, versus the reference caches. A small model that declines most of its
adjectives is telling you something the aggregate accuracy number hides.

In [ ]:
NOT_MENTIONED = ["not", "mentioned"]

REFERENCE = {
    "gpt4o-mini": "final_parsing_tokenized",
    "llama":      "llama_parsing_tokenized_clipped",
    "spacy":      "spacy_parsing_tokenized",
}
CANDIDATES = dict(REFERENCE)
CANDIDATES[MODEL] = PARSING_FOLDER


def field_report(folder, split="val"):
    path = f"data_parsing/{folder}/tokenized_parsed_result_{split}.json"
    if not os.path.isfile(path):
        return None
    flat = {}
    for scene, objs in json.load(open(path)).items():
        for obj, anns in objs.items():
            for ann, fields in anns.items():
                flat[f"{scene}|{obj}|{ann}"] = fields
    out = {"n": len(flat)}
    for field in ("target", "adjectives", "neighbors"):
        empty = sum(1 for v in flat.values() if v[field] == NOT_MENTIONED)
        out[field] = 100.0 * empty / max(len(flat), 1)
    return out


rows = []
for name, folder in CANDIDATES.items():
    report = field_report(folder)
    if report is None:
        rows.append([name, folder, "not built", "-", "-", "-"])
        continue
    rows.append([name, folder, report["n"],
                 f"{report['target']:.1f}%", f"{report['adjectives']:.1f}%",
                 f"{report['neighbors']:.1f}%"])

html = ["<table style='border-collapse:collapse;font-size:13px'>",
        "<tr>" + "".join(f"<th style='border-bottom:2px solid #444;padding:4px 10px;"
                         f"text-align:left'>{h}</th>"
                         for h in ("parser", "folder", "n",
                                   "target declined", "adjectives declined",
                                   "neighbors declined")) + "</tr>"]
for r in rows:
    weight = "font-weight:bold;" if r[0] == MODEL else ""
    html.append("<tr>" + "".join(
        f"<td style='border-bottom:1px solid #ddd;padding:4px 10px;{weight}'>{c}</td>"
        for c in r) + "</tr>")
html.append("</table>")
display(HTML("".join(html)))
display(Markdown("_'declined' = the parser returned `not mentioned` for that field._"))

---
## 6 · Score the target field against real ground truth

ScanRefer's `object_name` is free ground truth for `target`, so this one field can be
scored automatically over the whole split. This is the number that answers "is a 0.5B
model enough?".

Reference points already measured: LLaMA-3 **83.53%** exact, GPT-4o-mini **82.30%**,
spaCy **73.12%**.

In [ ]:
!python experiments/ablation/parsers/eval_parser_target_accuracy.py \
    --splits train --parsed-dir {TOK_DIR} --tag {MODEL}

---
## 7 · Compare all four fields against the other parsers

The two fields with no ground truth need the reference-free comparison — coverage,
faithfulness and agreement — which is what `parse_field_comparison.py` does. Adding the
new cache to that comparison places it beside GPT, LLaMA and spaCy directly.

In [ ]:
!python experiments/analysis/parse_field_comparison.py \
    --parse gpt4o-mini=final_parsing_tokenized \
    --parse llama=llama_parsing_tokenized_clipped \
    --parse spacy=spacy_parsing_tokenized \
    --parse {MODEL}={PARSING_FOLDER} \
    --split val

In [ ]:
from IPython.display import Image

png = "outputs/analysis/parse_field_comparison/parse_field_comparison.png"
if os.path.isfile(png):
    display(Image(filename=png))
else:
    display(Markdown(f"`{png}` not found — did the cell above run?"))

---
## 8 · Train on it

The cache is now a drop-in replacement for any other parse cache — the language module and
the fusion network are byte-identical across arms, so the only thing that changed is the
parse.

In [ ]:
print("Direct:")
print(f"    python scripts/ScanRefer_train.py \\")
print(f"        --parsing_folder {PARSING_FOLDER} \\")
print(f"        --use_cached_scenes --tag ABL-PARSER-SMALLLM \\")
print(f"        --use_color --use_normal")
print()
print("Or through the runner, which also handles the warm start:")
print("    python experiments/ablation/runners/run_parser_smalllm.py")
print()
print(f"NOTE: run_parser_smalllm.py has PARSING_FOLDER hardcoded to")
print(f"      'smalllm_parsing_tokenized'. For this cache, either edit that line to")
print(f"      '{PARSING_FOLDER}' or use the direct command above.")

### What was produced

| Path | Contents |
|---|---|
| `data_parsing/smalllm_<model>_parsing/` | raw phrase strings, GPT-cache shaped |
| `data_parsing/smalllm_<model>_parsing_tokenized/` | **what training reads** |
| `.../parse_run_summary.json` | model, device, prompt, per-split stats, malformed rate |
| `outputs/parser_eval/target_accuracy_<model>.json` | target accuracy vs `object_name` |

### Report in the manuscript

- the **malformed-output rate** from `parse_run_summary.json` (Reviewer #4 comment 3)
- **target accuracy** beside the other three parsers (Reviewer #3 comment 1)
- that parsing is **offline** — it contributes zero to per-query grounding latency
- the model id and that decoding was **greedy**, so the parse is reproducible